# The Decision Ladder: Prompt, RAG, or Fine-Tune?

**Phase 09** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-09/01-decision-ladder.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # Running locally — set OPENAI_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("OPENAI_API_KEY")))

## The Problem

A team ships a customer support chatbot backed by GPT-4. The answers are technically correct but sound robotic and generic - not like the company's brand voice. Someone suggests fine-tuning. Another person suggests just adding a system prompt. A third says to index the internal tone guide and use RAG. Three reasonable engineers, three different answers, and a $20,000 fine-tuning run on the table.

This decision gets made wrong constantly. Teams reach for fine-tuning out of instinct because it sounds like the most "AI-native" solution. They spend weeks building a dataset, pay for a training run, and ship a model that is not measurably better than a well-crafted system prompt would have been.
...

## The Concept

### The Five Rungs

```
COST ↑  DATA NEEDED ↑  TIME TO VALUE ↓
|
|  [5] Train from Scratch ..... millions of examples, months, tens of millions of $
|
|  [4] Fine-Tune .............. 100-10k examples, days-weeks, $50-$10k
|
|  [3] RAG .................... documents, hours, $0 extra per query
|
|  [2] Few-Shot Prompting ..... 3-20 examples in context, minutes, $0 extra
|
|  [1] Better System Prompt ... zero examples, minutes, $0 extra
|
COST ↓  DATA NEEDED ↓  TIME TO VALUE ↑
```

Always start at rung 1. Move up only when the current rung demonstrably fails. Moving from rung 3 to rung 4 requires...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
# code/main.py
# Dependencies: none (stdlib only)
# Usage: python main.py

from __future__ import annotations
import json
import sys
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class LadderQuestion:
    """A single question in the decision ladder."""
    id: str
    text: str
    yes_rung: Optional[str]  # rung to recommend if yes
    yes_label: str           # human label for the yes outcome
    no_continue: bool        # if True, continue to next question on no


LADDER_QUESTIONS: list[LadderQuestion] = [
    LadderQuestion(
        id="q1",
        text=(
            "Does the current system prompt clearly specify the persona, tone, "
            "format, and constraints? Have you tested at least 10 different phrasings "
            "of the instruction?"
        ),
        yes_rung=None,
        yes_label="Prompt engineering is not yet exhausted. Improve the system prompt first.",
        no_continue=True,
    ),
    LadderQuestion(
        id="q2",
        text=(
            "Have you tried adding 5-20 few-shot examples directly in the prompt "
            "that demonstrate the desired input/output pattern?"
        ),
        yes_rung=None,
        yes_label="Few-shot prompting is not yet exhausted. Add examples to the prompt first.",
        no_continue=True,
    ),
    LadderQuestion(
        id="q3",
        text=(
            "Is the problem primarily a KNOWLEDGE gap (missing facts, stale info, "
            "private documents the model was not trained on)? "
            "If yes, RAG is the right tool."
        ),
        yes_rung="RAG",
        yes_label=(
            "Use RAG. Index your knowledge base and retrieve relevant context at query time. "
            "Fine-tuning will not help here because it does not add new knowledge."
        ),
        no_continue=True,
    ),
    LadderQuestion(
        id="q4",
        text=(
            "Is the problem a BEHAVIOR gap: consistent output format, specialized vocabulary, "
            "brand tone, or latency/cost requirements that demand a smaller model? "
            "AND do you have at least 100 high-quality input/output examples?"
        ),
        yes_rung="FINE-TUNE",
        yes_label=(
            "Fine-tuning is justified. Build a curated dataset (see Lesson 02), "
            "start with the managed API (Lesson 03), and evaluate against your baseline (Lesson 05)."
        ),
        no_continue=True,
    ),
    LadderQuestion(
        id="q5",
        text=(
            "Are you building something with no available pretrained foundation - "
            "a completely novel domain with no overlap with any existing model? "
            "(This is extremely rare in practice.)"
        ),
        yes_rung="TRAIN-FROM-SCRATCH",
        yes_label=(
            "Training from scratch may be warranted, but verify that no existing model "
            "covers your domain first. This requires millions of examples, significant compute, "
            "and an ML research team."
        ),
        no_continue=False,
    ),
]


@dataclass
class EvaluationResult:
    """The result of running the decision ladder."""
    recommendation: str
    rung: int
    reasoning: str
    answers: list[dict] = field(default_factory=list)


RUNG_LABELS = {
    "PROMPT": 1,
    "FEW-SHOT": 2,
    "RAG": 3,
    "FINE-TUNE": 4,
    "TRAIN-FROM-SCRATCH": 5,
}


def ask_question(question: LadderQuestion) -> bool:
    """Ask a single ladder question and return True for yes, False for no."""
    print(f"\n{'='*60}")
    print(f"Question: {question.text}")
    print(f"{'='*60}")
    while True:
        answer = input("Answer [y/n]: ").strip().lower()
        if answer in ("y", "yes"):
            return True
        if answer in ("n", "no"):
            return False
        print("Please answer y or n.")


def run_ladder(problem: str) -> EvaluationResult:
    """Walk through the decision ladder and return a recommendation."""
    print(f"\nProblem: {problem}")
    print("\nWorking through the decision ladder from cheapest to most expensive...\n")

    answers = []

    for i, question in enumerate(LADDER_QUESTIONS):
        answered_yes = ask_question(question)
        answers.append({"question_id": question.id, "answer": "yes" if answered_yes else "no"})

        if answered_yes and question.yes_rung:
            # Positive branch: this rung is the recommendation
            rung_num = RUNG_LABELS.get(question.yes_rung, 4)
            return EvaluationResult(
                recommendation=question.yes_rung,
                rung=rung_num,
                reasoning=question.yes_label,
                answers=answers,
            )

        if answered_yes and not question.yes_rung:
            # Not yet exhausted this level - stay here
            return EvaluationResult(
                recommendation="PROMPT" if i == 0 else "FEW-SHOT",
                rung=i + 1,
                reasoning=question.yes_label,
                answers=answers,
            )

        # answered no: continue to next question

    # If we get through all questions with all no answers, default to prompting review
    return EvaluationResult(
        recommendation="REVISIT",
        rung=0,
        reasoning=(
            "Could not determine the right rung. Re-examine whether the problem is clearly "
            "defined. A fuzzy problem statement leads to the wrong tool choice."
        ),
        answers=answers,
    )


def print_result(result: EvaluationResult) -> None:
    """Print the evaluation result in a readable format."""
    print(f"\n{'='*60}")
    print("DECISION LADDER RESULT")
    print(f"{'='*60}")
    print(f"Recommendation: {result.recommendation} (Rung {result.rung})")
    print(f"\nReasoning:\n{result.reasoning}")
    print(f"\nYour answers: {json.dumps(result.answers, indent=2)}")


THREE_SCENARIOS = [
    {
        "name": "Customer tone matching",
        "description": (
            "The support chatbot answers correctly but sounds generic and corporate. "
            "The brand voice should be warm, direct, and human. "
            "We have 200 examples of 'bad' vs 'good' tone responses."
        ),
    },
    {
        "name": "Medical terminology extraction",
        "description": (
            "We need to extract ICD-10 codes from clinical notes. "
            "The base model gets common codes right but misses rare codes "
            "and uses wrong terminology for subspecialty conditions."
        ),
    },
    {
        "name": "FAQ answering",
        "description": (
            "Customers ask questions answered in our 500-page product manual. "
            "The base model makes up answers when it does not know. "
            "We have the manual but have not indexed it anywhere."
        ),
    },
]


def run_demo_scenarios() -> None:
    """Demonstrate the ladder on three canned scenarios without interactive input."""
    print("\nDEMO MODE: Running three scenarios with predetermined answers\n")

    scenario_answers = [
        # Customer tone matching: prompting exhausted, few-shot exhausted, not a knowledge gap, behavior gap with examples
        [False, False, False, True],
        # Medical terminology: prompting exhausted, few-shot exhausted, knowledge + behavior gap - RAG first
        [False, False, True, None],
        # FAQ answering: prompting not exhausted yet (no RAG in place)
        [False, False, True, None],
    ]

    for i, (scenario, answers) in enumerate(zip(THREE_SCENARIOS, scenario_answers)):
        print(f"\n{'#'*60}")
        print(f"Scenario {i+1}: {scenario['name']}")
        print(f"Problem: {scenario['description']}")
        print(f"{'#'*60}")

        # Simulate the ladder with predetermined answers
        result_answers = []
        recommendation = "REVISIT"
        rung = 0
        reasoning = ""

        for j, (question, answer) in enumerate(zip(LADDER_QUESTIONS, answers)):
            if answer is None:
                break
            result_answers.append({"question_id": question.id, "answer": "yes" if answer else "no"})

            if answer and question.yes_rung:
                recommendation = question.yes_rung
                rung = RUNG_LABELS.get(question.yes_rung, 4)
                reasoning = question.yes_label
                break
            if answer and not question.yes_rung:
                recommendation = "PROMPT" if j == 0 else "FEW-SHOT"
                rung = j + 1
                reasoning = question.yes_label
                break

        result = EvaluationResult(
            recommendation=recommendation,
            rung=rung,
            reasoning=reasoning,
            answers=result_answers,
        )
        print_result(result)

### Demo

In [ ]:
if "--demo" in sys.argv:
    run_demo_scenarios()
else:
    print("Decision Ladder: Prompt, RAG, or Fine-Tune?")
    print("--------------------------------------------")
    problem = input("Describe your problem in one sentence: ").strip()
    if not problem:
        problem = "Unspecified problem"
    result = run_ladder(problem)
    print_result(result)

---

## Self-Check

Answer these without running code first:

**1. What should you recommend, and why?**

- A. Start the fine-tune immediately - 50 examples is enough for tone adaptation
- B. Try a detailed system prompt and few-shot examples first - prompting is not yet exhausted and 50 examples is too few to fine-tune reliably
- C. Use RAG - index the brand voice guide and retrieve tone examples at query time
- D. Upgrade to a larger model - the robotic tone is a capability limitation

**2. Which tool is most appropriate as the first intervention?**

- A. Fine-tune the base model on the 10,000 case studies
- B. Add the case studies to a vector store and use RAG to retrieve relevant cases at query time
- C. Write a detailed system prompt that tells the model to be accurate about rare diseases
- D. Train a model from scratch on the proprietary database

**3. Is fine-tuning justified here? What is the key indicator?**

- A. No - the 15% failure rate suggests the model is capable and prompting needs more iteration
- B. No - this is a knowledge gap and RAG with schema examples would solve it
- C. Yes - strict output format that cannot be reliably achieved with prompting + few-shot is a documented justification for fine-tuning
- D. Yes - any structured output task always requires fine-tuning to be reliable

**4. What is the recommended approach?**

- A. Fine-tune only - the vocabulary requirement is purely behavioral
- B. RAG only - index the glossary and retrieve the correct terms at query time
- C. RAG first (index the glossary), then evaluate whether fine-tuning on the 150 annotated examples is still needed for the residual vocabulary gap
- D. Few-shot prompting with 20 examples from the glossary - no need for RAG or fine-tuning

**5. What is the decision ladder's diagnosis of this situation?**

- A. More data will fix it - fine-tuning is the right tool, just needs scale
- B. Fine-tuning is amplifying existing math capability, not creating new reasoning ability. Use a more capable base model.
- C. Switch to RAG - index solved math problems and retrieve similar examples at query time
- D. The prompting has not been exhausted - add chain-of-thought instructions to the system prompt

**6. How should you evaluate this argument?**

- A. Valid - a fine-tuned model is a defensible proprietary asset that competitors cannot copy
- B. Partially valid - the fine-tune itself is not the moat, but the dataset it was trained on is
- C. Invalid - all fine-tunes eventually become open source so there is no long-term advantage
- D. Valid only if the fine-tune achieves >10% improvement over the baseline

**7. What is the critical question the decision ladder surfaces before approving this fine-tune?**

- A. 80,000 is too many examples - large datasets cause overfitting
- B. Are these transcripts already formatted as input/output pairs with quality labels, or are they raw conversations that need curation?
- C. Fine-tuning APIs have a maximum of 10,000 examples, so the dataset must be downsampled
- D. 80,000 examples guarantees success - proceed immediately

**8. What should happen before the fine-tuning job starts?**

- A. Start the fine-tune - 120 examples meets the minimum threshold
- B. Split the 120 examples into train/validation/test sets, establish a baseline eval score from the prompt-only model, and only proceed if the held-out test set is large enough to measure improvement
- C. Collect more examples until you have at least 500 before spending on training compute
- D. Run few-shot prompting with all 120 examples in context first - if that works, no fine-tuning needed

_Answers are in `checks.json` in the lesson directory._